# SCA — REST API

Esta API conecta los tres equipos del proyecto:

```
 Drag & Drop UI (Diseño)          Forms UI (Formularios)
        │                                  │
        └──────────────┬───────────────────┘
                       │  HTTP REST
               ┌───────▼────────┐
               │   SCA API      │  ← este notebook
               │   (FastAPI)    │
               └───────┬────────┘
                       │ SQLAlchemy
               ┌───────▼────────┐
               │  PostgreSQL    │
               │   (sca_db)     │
               └───────┬────────┘
                       │ sync engine (main.py)
               ┌───────▼────────┐
               │ Google Sheets  │
               └────────────────┘
```

**Regla central:** toda escritura de la UI marca `is_dirty=TRUE` y `last_modified_by='app'`.
El sync engine detecta esos registros y los sube automáticamente a Sheets en el siguiente ciclo (cada 120 s).

---
## Endpoints

| Método | Ruta | Descripción |
|--------|------|-------------|
| GET | `/corridas` | Lista corridas del día |
| GET | `/corridas/{serie}` | Detalle de una corrida |
| PUT | `/corridas/{serie}` | Actualiza corrida desde la UI |
| GET | `/registros` | Vista CENTRAL (todos los camiones activos) |
| GET | `/registros/{serie}` | Registro + checklist de un camión |
| PUT | `/registros/{serie}` | Actualiza ubicación/avance desde la UI |
| GET | `/movimientos` | Lista movimientos (filtros: area, serie, fecha) |
| POST | `/movimientos` | Registra entrada de camión a un área |
| PUT | `/movimientos/{id}` | Actualiza movimiento |
| PUT | `/movimientos/{id}/completar` | Marca movimiento como completado |
| GET | `/areas` | Lookup de áreas |
| GET | `/tipos-camion` | Lookup de tipos de camión |
| GET | `/tiempos` | Tiempos del día |
| GET | `/kpis` | Último snapshot de KPIs |
| POST | `/turno/archivar` | Cierra y archiva un turno |

## 1. Instalación de dependencias

In [1]:
%pip install fastapi "uvicorn[standard]" nest-asyncio httpx --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Imports y configuración de base de datos

In [ ]:
import sys
import os
import json
import threading
from datetime import datetime, date
from typing import Optional
from contextlib import contextmanager

import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from sqlalchemy import create_engine, text

sys.path.insert(0, os.getcwd())
from config import DATABASE_URL
from db_client import AREA_NOMBRE_DB

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

@contextmanager
def db():
    
    with engine.connect() as conn:
        with conn.begin():
            yield conn

def row_to_dict(row):
    return dict(row._mapping)

def rows_to_list(rows):
    return [row_to_dict(r) for r in rows]

print("OK — conexión a DB configurada:", DATABASE_URL.split("@")[-1])

## 3. Modelos Pydantic (esquema de requests)

Estos modelos validan el JSON que mandan los frontends.
Todos los campos son opcionales para soportar actualizaciones parciales (PATCH semántico via PUT).

In [2]:
class CorrridaUpdate(BaseModel):
    hora_corrida:   Optional[str]  = None
    hora_salida:    Optional[str]  = None
    tipo_nombre:    Optional[str]  = None
    need_recepcion: Optional[int]  = None
    need_desfogue:  Optional[int]  = None
    need_diesel:    Optional[int]  = None
    need_adblue:    Optional[int]  = None
    need_lav_ext:   Optional[int]  = None
    need_lav_int:   Optional[int]  = None
    need_taller:    Optional[int]  = None

class RegistroUpdate(BaseModel):
    ubicacion_texto: Optional[str]   = None
    tiempo_restante: Optional[float] = None
    avance:          Optional[float] = None

class MovimientoCreate(BaseModel):
    serie:        int
    area_nombre:  str
    hora_entrada: Optional[str] = None

class MovimientoUpdate(BaseModel):
    hora_salida:   Optional[str]   = None
    completado:    Optional[bool]  = None
    duracion_dias: Optional[float] = None

class ArchivarTurnoRequest(BaseModel):
    turno:      int
    usuario_id: int

print("OK — modelos Pydantic cargados")

OK — modelos Pydantic cargados


## 4. Aplicación FastAPI + CORS

CORS abierto (`allow_origins=["*"]`) para que ambos frontends (Diseño y Formularios)
puedan llamar la API sin importar el puerto desde el que corran.

In [3]:
app = FastAPI(
    title="SCA API",
    version="1.0.0",
    description="API REST para el Sistema de Control de Autobuses — ADO Oaxaca",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

print("OK — app FastAPI creada")

OK — app FastAPI creada


## 5. Endpoints — Corridas (`/corridas`)

Tabla `corridas` ↔ Sheet `PLANEACION`. Clave única: `(fecha, serie)`.

In [ ]:
CORRIDA_COLUMNS = (
    "c.id, c.date AS fecha, c.serial_number AS serie, c.type_id AS tipo_id,"
    " c.scheduled_time AS hora_corrida, c.departure_time AS hora_salida,"
    " c.needs_reception AS need_recepcion, c.needs_drainage AS need_desfogue,"
    " c.needs_diesel AS need_diesel, c.needs_adblue AS need_adblue,"
    " c.needs_ext_wash AS need_lav_ext, c.needs_int_wash AS need_lav_int,"
    " c.needs_workshop AS need_taller, c.sheets_row, c.is_dirty,"
    " c.last_modified_by, c.last_modified_at, c.sheets_synced_at"
)

# Pydantic (español) -> columna real en "trips"
CORRIDA_FIELD_DB = {
    "hora_corrida":   "scheduled_time",
    "hora_salida":    "departure_time",
    "need_recepcion": "needs_reception",
    "need_desfogue":  "needs_drainage",
    "need_diesel":    "needs_diesel",
    "need_adblue":    "needs_adblue",
    "need_lav_ext":   "needs_ext_wash",
    "need_lav_int":   "needs_int_wash",
    "need_taller":    "needs_workshop",
}

@app.get("/corridas", summary="Lista corridas del día")
def get_corridas(fecha: Optional[str] = Query(default=None, description="YYYY-MM-DD, default = hoy")):
    target = fecha or str(date.today())
    with db() as conn:
        rows = conn.execute(text(
            f"SELECT {CORRIDA_COLUMNS}, tc.name AS tipo_nombre"
            " FROM trips c JOIN bus_types tc ON c.type_id = tc.id"
            " WHERE c.date = :f ORDER BY c.serial_number"
        ), {"f": target}).fetchall()
    return rows_to_list(rows)

@app.get("/corridas/{serie}", summary="Detalle de una corrida")
def get_corrida(serie: int, fecha: Optional[str] = Query(default=None)):
    target = fecha or str(date.today())
    with db() as conn:
        row = conn.execute(text(
            f"SELECT {CORRIDA_COLUMNS}, tc.name AS tipo_nombre"
            " FROM trips c JOIN bus_types tc ON c.type_id = tc.id"
            " WHERE c.date = :f AND c.serial_number = :s"
        ), {"f": target, "s": serie}).fetchone()
    if not row:
        raise HTTPException(404, f"Corrida serie={serie} no encontrada para fecha={target}")
    return row_to_dict(row)

@app.put("/corridas/{serie}", summary="Actualiza corrida desde la UI")
def update_corrida(serie: int, body: CorrridaUpdate):
    target = str(date.today())
    updates = {k: v for k, v in body.model_dump().items() if v is not None}
    if not updates:
        raise HTTPException(400, "No hay campos para actualizar")

    with db() as conn:
        existing = conn.execute(text(
            "SELECT id, type_id FROM trips WHERE date=:f AND serial_number=:s"
        ), {"f": target, "s": serie}).fetchone()
        if not existing:
            raise HTTPException(404, f"Corrida serie={serie} no encontrada")

        tipo_id = existing.type_id
        if "tipo_nombre" in updates:
            tipo_row = conn.execute(text(
                "SELECT id FROM bus_types WHERE name=:n"
            ), {"n": updates.pop("tipo_nombre")}).fetchone()
            if not tipo_row:
                raise HTTPException(400, "tipo_nombre no válido")
            tipo_id = tipo_row[0]

        set_clauses = ", ".join(f"{CORRIDA_FIELD_DB[k]}=:{k}" for k in updates)
        params = {**updates, "tipo_id": tipo_id, "f": target, "s": serie, "ts": datetime.now()}

        conn.execute(text(
            f"UPDATE trips SET {set_clauses}, type_id=:tipo_id,"
            " is_dirty=true, last_modified_by='app', last_modified_at=:ts"
            " WHERE date=:f AND serial_number=:s"
        ), params)

    return {"ok": True, "serie": serie, "fecha": target}

print("OK — endpoints /corridas registrados")

## 6. Endpoints — Registros (`/registros`)

Tabla `registros` + `checklist_areas` ↔ Sheet `CENTRAL`.
Representa el estado actual de cada camión en el patio.

In [ ]:
REGISTRO_COLUMNS = (
    "id, date AS fecha, serial_number AS serie, type_id AS tipo_id, shift AS turno,"
    " is_active AS activo, registration_time AS hora_registro, location_text AS ubicacion_texto,"
    " time_remaining AS tiempo_restante, progress AS avance, sheets_row, is_dirty,"
    " last_modified_by, last_modified_at, sheets_synced_at"
)

# Pydantic (español) -> columna real en "records"
REGISTRO_FIELD_DB = {
    "ubicacion_texto": "location_text",
    "tiempo_restante":  "time_remaining",
    "avance":           "progress",
}

@app.get("/registros", summary="Vista CENTRAL — todos los camiones activos hoy")
def get_registros(fecha: Optional[str] = Query(default=None)):
    target = fecha or str(date.today())
    with db() as conn:
        rows = conn.execute(text(
            f"SELECT {REGISTRO_COLUMNS} FROM records WHERE date = :f AND is_active = true ORDER BY serial_number"
        ), {"f": target}).fetchall()
    return rows_to_list(rows)

@app.get("/registros/{serie}", summary="Registro + checklist de un camión")
def get_registro(serie: int, fecha: Optional[str] = Query(default=None)):
    target = fecha or str(date.today())
    with db() as conn:
        row = conn.execute(text(
            f"SELECT {REGISTRO_COLUMNS} FROM records WHERE date = :f AND serial_number = :s AND is_active = true"
        ), {"f": target, "s": serie}).fetchone()
        if not row:
            raise HTTPException(404, f"Registro serie={serie} no encontrado para {target}")

        checklist = conn.execute(text(
            "SELECT ca.id, ca.record_id AS registro_id, ca.area_id,"
            " ca.is_checked AS completada, ca.quantity AS espacios, a.name AS area_nombre"
            " FROM area_checklists ca JOIN area a ON ca.area_id = a.id"
            " WHERE ca.record_id = :rid ORDER BY a.name"
        ), {"rid": row.id}).fetchall()

    result = row_to_dict(row)
    result["checklist"] = rows_to_list(checklist)
    return result

@app.put("/registros/{serie}", summary="Actualiza ubicación/avance desde la UI")
def update_registro(serie: int, body: RegistroUpdate):
    target = str(date.today())
    updates = {k: v for k, v in body.model_dump().items() if v is not None}
    if not updates:
        raise HTTPException(400, "No hay campos para actualizar")

    with db() as conn:
        existing = conn.execute(text(
            "SELECT id FROM records WHERE date=:f AND serial_number=:s AND is_active=true"
        ), {"f": target, "s": serie}).fetchone()
        if not existing:
            raise HTTPException(404, f"Registro serie={serie} no encontrado")

        set_clauses = ", ".join(f"{REGISTRO_FIELD_DB[k]}=:{k}" for k in updates)
        params = {**updates, "f": target, "s": serie, "ts": datetime.now()}

        conn.execute(text(
            f"UPDATE records SET {set_clauses},"
            " is_dirty=true, last_modified_by='app', last_modified_at=:ts"
            " WHERE date=:f AND serial_number=:s AND is_active=true"
        ), params)

    return {"ok": True, "serie": serie, "fecha": target}

print("OK — endpoints /registros registrados")

## 7. Endpoints — Movimientos (`/movimientos`)

Tabla `movimientos` ↔ Sheets `DIESEL`, `ADDBLUE`, `TALLER`, `DESFOGUE`, `LAVADO EXTERIOR`, `LAVADO INTERIOR`.

Un movimiento = un camión en un área de servicio.
El flujo normal de la UI es:
1. `POST /movimientos` → camión entra al área
2. `PUT /movimientos/{id}/completar` → servicio terminado
3. El sync engine lo sube a la hoja correspondiente.

In [ ]:
# Pydantic (español) -> columna real en "movements"
MOVIMIENTO_FIELD_DB = {
    "hora_salida":   "exit_time",
    "completado":    "is_completed",
    "duracion_dias": "duration_days",
}

@app.get("/movimientos", summary="Lista movimientos con filtros opcionales")
def get_movimientos(
    area:  Optional[str] = Query(default=None, description="Nombre del área ej: DIESEL"),
    serie: Optional[int] = Query(default=None, description="Número de serie del camión"),
    fecha: Optional[str] = Query(default=None, description="YYYY-MM-DD, default = hoy"),
):
    target = fecha or str(date.today())
    q = (
        "SELECT m.id, m.record_id AS registro_id, m.area_id, m.serial_number AS serie,"
        " m.date AS fecha, m.entry_time AS hora_entrada, m.exit_time AS hora_salida,"
        " m.is_completed AS completado, m.duration_days AS duracion_dias,"
        " m.sheets_row, m.is_dirty, m.last_modified_by, m.last_modified_at, m.sheets_synced_at,"
        " a.name AS area_nombre"
        " FROM movements m JOIN area a ON m.area_id = a.id"
        " WHERE m.date = :f"
    )
    params = {"f": target}
    if area:
        q += " AND a.name = :area"
        params["area"] = AREA_NOMBRE_DB.get(area, area)
    if serie:
        q += " AND m.serial_number = :serie"
        params["serie"] = serie
    q += " ORDER BY m.entry_time"

    with db() as conn:
        rows = conn.execute(text(q), params).fetchall()
    return rows_to_list(rows)

@app.post("/movimientos", status_code=201, summary="Registra entrada de camión a un área")
def create_movimiento(body: MovimientoCreate):
    with db() as conn:
        area_row = conn.execute(text(
            "SELECT id FROM area WHERE name = :n"
        ), {"n": AREA_NOMBRE_DB.get(body.area_nombre, body.area_nombre)}).fetchone()
        if not area_row:
            raise HTTPException(400, f"Área '{body.area_nombre}' no existe. "
                                     f"Consulta GET /areas para ver las disponibles.")
        area_id = area_row[0]

        registro = conn.execute(text(
            "SELECT id FROM records WHERE date=:f AND serial_number=:s AND is_active=true"
        ), {"f": date.today(), "s": body.serie}).fetchone()

        if not registro:
            r = conn.execute(text(
                "INSERT INTO records (date, serial_number, is_active, is_dirty, last_modified_by, last_modified_at)"
                " VALUES (:f, :s, true, true, 'app', NOW()) RETURNING id"
            ), {"f": date.today(), "s": body.serie}).fetchone()
            registro_id = r[0] if r else conn.execute(text(
                "SELECT id FROM records WHERE date=:f AND serial_number=:s"
            ), {"f": date.today(), "s": body.serie}).fetchone()[0]
        else:
            registro_id = registro[0]

        now = datetime.now()
        hora_entrada = body.hora_entrada or now.strftime("%H:%M:%S")

        result = conn.execute(text(
            "INSERT INTO movements"
            " (record_id, area_id, serial_number, date, entry_time, is_dirty, last_modified_by, last_modified_at)"
            " VALUES (:rid, :aid, :serie, CURRENT_DATE, :hora_entrada, true, 'app', :ts)"
            " RETURNING id"
        ), {
            "rid": registro_id, "aid": area_id,
            "serie": body.serie, "hora_entrada": hora_entrada, "ts": now,
        }).fetchone()

    return {"ok": True, "id": result[0], "serie": body.serie, "area": body.area_nombre}

@app.put("/movimientos/{mov_id}", summary="Actualiza datos de un movimiento")
def update_movimiento(mov_id: int, body: MovimientoUpdate):
    updates = {k: v for k, v in body.model_dump().items() if v is not None}
    if not updates:
        raise HTTPException(400, "No hay campos para actualizar")

    with db() as conn:
        if not conn.execute(text(
            "SELECT id FROM movements WHERE id=:id"
        ), {"id": mov_id}).fetchone():
            raise HTTPException(404, f"Movimiento id={mov_id} no encontrado")

        set_clauses = ", ".join(f"{MOVIMIENTO_FIELD_DB[k]}=:{k}" for k in updates)
        conn.execute(text(
            f"UPDATE movements SET {set_clauses},"
            " is_dirty=true, last_modified_by='app', last_modified_at=:ts"
            " WHERE id=:id"
        ), {**updates, "id": mov_id, "ts": datetime.now()})

    return {"ok": True, "id": mov_id}

@app.put("/movimientos/{mov_id}/completar", summary="Marca un movimiento como completado")
def completar_movimiento(mov_id: int):
    with db() as conn:
        if not conn.execute(text(
            "SELECT id FROM movements WHERE id=:id"
        ), {"id": mov_id}).fetchone():
            raise HTTPException(404, f"Movimiento id={mov_id} no encontrado")

        now = datetime.now()
        conn.execute(text(
            "UPDATE movements SET is_completed=true, exit_time=:ts,"
            " is_dirty=true, last_modified_by='app', last_modified_at=:ts"
            " WHERE id=:id"
        ), {"id": mov_id, "ts": now})

    return {"ok": True, "id": mov_id, "completado": True}

print("OK — endpoints /movimientos registrados")

## 8. Endpoints — Lookups, Tiempos y KPIs

In [ ]:
@app.get("/areas", summary="Lista de áreas de servicio")
def get_areas():
    reverse_area = {v: k for k, v in AREA_NOMBRE_DB.items()}
    with db() as conn:
        rows = conn.execute(text("SELECT id, name FROM area ORDER BY name")).fetchall()
    return [{"id": r.id, "nombre": reverse_area.get(r.name, r.name)} for r in rows]

@app.get("/tipos-camion", summary="Lista de tipos de camión")
def get_tipos():
    with db() as conn:
        rows = conn.execute(text("SELECT id, name AS nombre FROM bus_types ORDER BY name")).fetchall()
    return rows_to_list(rows)

@app.get("/tiempos", summary="Tiempos de servicio — Sheet TIEMPOS")
def get_tiempos():
    with db() as conn:
        rows = conn.execute(text(
            "SELECT id, serial_number AS serie, entry_time AS hora_entrada,"
            " is_completed AS completado, exit_time_num AS hora_salida_num,"
            " entry_time_num AS hora_entrada_num, duration_days AS duracion_dias"
            " FROM times ORDER BY entry_time"
        )).fetchall()
    return rows_to_list(rows)

@app.get("/kpis", summary="Último snapshot de KPIs")
def get_kpis():
    with db() as conn:
        row = conn.execute(text(
            "SELECT id, captured_at,"
            " total_buses AS total_camiones, buses_needing_workshop AS camiones_necesitan_taller,"
            " released_buses AS camiones_liberados, serviced_buses AS camiones_atendidos,"
            " buses_in_bays AS camiones_en_andenes, foreign_buses AS camiones_foraneos,"
            " buses_needing_wash AS camiones_necesitan_lavado,"
            " priority_units_completed AS unidades_prioritarias_cumplidas,"
            " mechanics_count AS num_mecanicos, total_bay_capacity AS capacidad_total_andenes,"
            " pct_needing_workshop AS pct_necesitan_taller, pct_released AS pct_liberados,"
            " workload_per_mechanic AS carga_por_mecanico, bay_utilization AS utilizacion_andenes,"
            " foreign_bus_flow AS flujo_foraneos, pct_needing_wash AS pct_necesitan_lavado,"
            " priority_compliance AS cumplimiento_prioridad"
            " FROM kpis__snapshot ORDER BY id DESC LIMIT 1"
        )).fetchone()
    if not row:
        raise HTTPException(404, "No hay snapshots de KPIs todavía")
    return row_to_dict(row)

print("OK — endpoints lookups / tiempos / kpis registrados")

## 9. Endpoint — Cierre de turno (`/turno/archivar`)

Crea un snapshot JSON del turno y marca todos sus registros como `activo=false`.
Igual a `archivar_turno()` en `db_client.py` pero expuesto como REST.

In [ ]:
@app.post("/turno/archivar", summary="Cierra y archiva un turno")
def archivar_turno(body: ArchivarTurnoRequest):
    with db() as conn:
        registros = conn.execute(text(
            "SELECT row_to_json(r) FROM records r WHERE date=:f AND shift=:t"
        ), {"f": date.today(), "t": body.turno}).fetchall()

        movimientos = conn.execute(text(
            "SELECT row_to_json(m) FROM movements m"
            " WHERE m.date=:f"
            " AND m.record_id IN (SELECT id FROM records WHERE date=:f AND shift=:t)"
        ), {"f": date.today(), "t": body.turno}).fetchall()

        conn.execute(text(
            "INSERT INTO shift_closures (date, shift, closed_by, records_snapshot, movements_snapshot)"
            " VALUES (:f, :t, :u, CAST(:sr AS jsonb), CAST(:sm AS jsonb))"
        ), {
            "f": date.today(),
            "t": body.turno,
            "u": body.usuario_id,
            "sr": json.dumps([r[0] for r in registros]),
            "sm": json.dumps([m[0] for m in movimientos]),
        })

        conn.execute(text(
            "UPDATE records SET is_active=false WHERE date=:f AND shift=:t"
        ), {"f": date.today(), "t": body.turno})

    return {
        "ok": True,
        "turno": body.turno,
        "registros_archivados": len(registros),
        "movimientos_archivados": len(movimientos),
    }

print("OK — endpoint /turno/archivar registrado")

## 10. Correr el servidor

`nest_asyncio` permite correr `uvicorn` dentro del event loop de Jupyter.
El servidor corre en un **daemon thread** para que el notebook siga interactivo.

- **Docs interactivas (Swagger):** http://localhost:8000/docs
- **Docs alternativas (ReDoc):** http://localhost:8000/redoc

> Comparte la URL `http://<tu-IP>:8000` con los otros equipos mientras estés en la misma red.

In [ ]:
nest_asyncio.apply()

def _run_server():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    # loop="asyncio": uvicorn[standard] trae uvloop, y su política de event loop
    # no soporta correr en un hilo secundario junto con nest_asyncio (revienta con
    # "There is no current event loop in thread"). Forzar el loop estándar de
    # asyncio evita el choque.
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning", loop="asyncio")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

import time
time.sleep(1)

print("Servidor corriendo en:")
print("  http://localhost:8000")
print("  http://localhost:8000/docs   ← Swagger UI")
print("  http://localhost:8000/redoc  ← ReDoc")

## 11. Pruebas con `httpx`

Corre cada celda por separado para verificar que cada endpoint responde correctamente.
Estas mismas llamadas son la referencia que pueden usar los equipos de Diseño y Formularios.

In [10]:
import httpx
import json

BASE = "http://localhost:8000"

def pretty(r):
    print(f"  Status : {r.status_code}")
    try:
        print(f"  Body   : {json.dumps(r.json(), indent=2, default=str)[:600]}")
    except Exception:
        print(f"  Body   : {r.text[:300]}")

In [11]:
print("GET /areas")
pretty(httpx.get(f"{BASE}/areas"))

GET /areas


ConnectError: [Errno 111] Connection refused

In [ ]:
print("GET /tipos-camion")
pretty(httpx.get(f"{BASE}/tipos-camion"))

In [12]:
print("GET /corridas  (hoy)")
pretty(httpx.get(f"{BASE}/corridas"))

GET /corridas  (hoy)


ConnectError: [Errno 111] Connection refused

In [13]:
print("GET /registros  (hoy)")
pretty(httpx.get(f"{BASE}/registros"))

GET /registros  (hoy)


ConnectError: [Errno 111] Connection refused

In [14]:
print("GET /movimientos  (hoy, todos)")
pretty(httpx.get(f"{BASE}/movimientos"))

print("\nGET /movimientos?area=DIESEL")
pretty(httpx.get(f"{BASE}/movimientos", params={"area": "DIESEL"}))

GET /movimientos  (hoy, todos)


ConnectError: [Errno 111] Connection refused

In [15]:
print("POST /movimientos")
pretty(httpx.post(f"{BASE}/movimientos", json={
    "serie": 1001,
    "area_nombre": "DIESEL",
    "hora_entrada": "08:30:00",
}))

POST /movimientos


ConnectError: [Errno 111] Connection refused

In [16]:
MOV_ID = 1
print(f"PUT /movimientos/{MOV_ID}/completar")
pretty(httpx.put(f"{BASE}/movimientos/{MOV_ID}/completar"))

PUT /movimientos/1/completar


ConnectError: [Errno 111] Connection refused

In [17]:
SERIE = 1001
print(f"PUT /corridas/{SERIE}")
pretty(httpx.put(f"{BASE}/corridas/{SERIE}", json={
    "need_diesel": 1,
    "hora_corrida": "06:00:00",
}))

PUT /corridas/1001


ConnectError: [Errno 111] Connection refused

In [18]:
print("GET /kpis")
pretty(httpx.get(f"{BASE}/kpis"))

GET /kpis


ConnectError: [Errno 111] Connection refused

---
## Notas para los otros equipos

### Contrato de escritura
Todo endpoint `POST` / `PUT` automáticamente marca:
```json
{ "is_dirty": true, "last_modified_by": "app", "last_modified_at": "<timestamp>" }
```
El sync engine (`main.py`) detecta `is_dirty=true` y sube el cambio a Google Sheets en el siguiente ciclo (~2 min).

### Áreas válidas para `/movimientos`
Consultar siempre `GET /areas` para obtener los nombres exactos.
Los valores actuales del Sheet son: `DIESEL`, `ADDBLUE`, `TALLER`, `DESFOGUE`, `LAVADO EXTERIOR`, `LAVADO INTERIOR`.

### Documentación interactiva
Con el servidor corriendo, abrir **http://localhost:8000/docs** para ver y probar todos los endpoints desde el navegador.